# Búsqueda: de Arad a Bucharest

Russell & Norvig, *Artificial Intelligence: A Modern Approach*, 4ª edición, capítulo 3.

***
## Parte 1

Configuración inicial del problema

### 3.1.1 — El problema

Para abstrear el problema de busqueda debemos definir los siguientes elementos de este:

- **INITIAL**: Estado inicial
- **GOAL**: Objetivo
- **ACTIONS(s)**: Acciones que se pueden ejecutar en un estado `s`
- **RESULT(s, a)**: El estado al que se llega desde un estado `s` ejecutando la acción `a`
- **ACTION_COST(s, a, s')**: El costo de llegar al estado `s'` desde `s` mediante la acción `a`

In [1]:
class Problem:
    """Problema abstracto. Para definir uno concreto se heredan
    ACTIONS, RESULT y, si el costo no es uniforme, ACTION_COST."""

    def __init__(self, INITIAL=None, GOAL=None, **kwds):
        self.INITIAL = INITIAL
        self.GOAL = GOAL
        self.__dict__.update(**kwds)

    def ACTIONS(self, state):
        """Las acciones aplicables en `state`."""
        raise NotImplementedError

    def RESULT(self, state, action):
        """El estado que resulta de aplicar `action` en `state`."""
        raise NotImplementedError

    def ACTION_COST(self, s, action, s_prime):
        """El costo de ir de `s` a `s_prime` con `action`."""
        return 1

    def IS_GOAL(self, state):
        return state == self.GOAL

    def h(self, node):
        """Heuristica: costo estimado desde node.STATE hasta el objetivo.
        Por defecto 0, no informa nada."""
        return 0

    def __str__(self):
        return f"{type(self).__name__}({self.INITIAL!r}, {self.GOAL!r})"

### El mapa de Rumania

Carga del mapa

In [2]:
from collections import defaultdict


class Map:
    """Ciudades unidas por rutas con distancia."""

    def __init__(self, links, locations=None, straight_line_distance=None,
                 directed=False):
        self.distances = {}
        self.neighbors = defaultdict(list)
        for (city1, city2, distance) in links:
            self.distances[city1, city2] = distance
            self.neighbors[city1].append(city2)
            if not directed:
                self.distances[city2, city1] = distance
                self.neighbors[city2].append(city1)
        # Orden alfabetico: hace que la busqueda sea reproducible.
        for city in self.neighbors:
            self.neighbors[city].sort()
        self.locations = locations or {}
        self.straight_line_distance = straight_line_distance or {}

In [3]:
# Las 23 rutas del mapa, con su distancia en km.
links = [
    ("Arad", "Zerind", 75),          ("Arad", "Sibiu", 140),
    ("Arad", "Timisoara", 118),      ("Bucharest", "Fagaras", 211),
    ("Bucharest", "Pitesti", 101),   ("Bucharest", "Giurgiu", 90),
    ("Bucharest", "Urziceni", 85),   ("Craiova", "Drobeta", 120),
    ("Craiova", "Rimnicu", 146),     ("Craiova", "Pitesti", 138),
    ("Drobeta", "Mehadia", 75),      ("Eforie", "Hirsova", 86),
    ("Fagaras", "Sibiu", 99),        ("Hirsova", "Urziceni", 98),
    ("Iasi", "Vaslui", 92),          ("Iasi", "Neamt", 87),
    ("Lugoj", "Timisoara", 111),     ("Lugoj", "Mehadia", 70),
    ("Oradea", "Zerind", 71),        ("Oradea", "Sibiu", 151),
    ("Pitesti", "Rimnicu", 97),      ("Rimnicu", "Sibiu", 80),
    ("Urziceni", "Vaslui", 142),
]

# Distancia en linea recta hasta Bucharest: la heuristica h(n).
straight_line_distance = {
    "Arad": 366,    "Bucharest": 0,   "Craiova": 160,  "Drobeta": 242,
    "Eforie": 161,  "Fagaras": 176,   "Giurgiu": 77,   "Hirsova": 151,
    "Iasi": 226,    "Lugoj": 244,     "Mehadia": 241,  "Neamt": 234,
    "Oradea": 380,  "Pitesti": 100,   "Rimnicu": 193,  "Sibiu": 253,
    "Timisoara": 329, "Urziceni": 80, "Vaslui": 199,   "Zerind": 374,
}

# Posicion de cada ciudad, para dibujarlas.
locations = {
    "Arad": (91, 492),      "Bucharest": (400, 327),  "Craiova": (253, 288),
    "Drobeta": (165, 299),  "Eforie": (562, 293),     "Fagaras": (305, 449),
    "Giurgiu": (375, 270),  "Hirsova": (534, 350),    "Iasi": (473, 506),
    "Lugoj": (165, 379),    "Mehadia": (168, 339),    "Neamt": (406, 537),
    "Oradea": (131, 571),   "Pitesti": (320, 368),    "Rimnicu": (233, 410),
    "Sibiu": (207, 457),    "Timisoara": (94, 410),   "Urziceni": (456, 350),
    "Vaslui": (509, 444),   "Zerind": (108, 531),
}

romania = Map(links, locations, straight_line_distance)

### Configuración inicial del problema

In [4]:
class RouteProblem(Problem):
    """Ir de una ciudad a otra sobre un mapa."""

    def __init__(self, INITIAL, GOAL, map):
        super().__init__(INITIAL, GOAL, map=map)

    def ACTIONS(self, state):
        return self.map.neighbors[state]

    def RESULT(self, state, action):
        # La accion es la ciudad destino, siempre que sea vecina.
        return action if action in self.map.neighbors[state] else state

    def ACTION_COST(self, s, action, s_prime):
        return self.map.distances[s, s_prime]

    def h(self, node):
        """Distancia en linea recta hasta el objetivo (figura 3.22)."""
        return self.map.straight_line_distance.get(node.STATE, 0)

In [5]:
problem = RouteProblem("Arad", "Bucharest", map=romania)

print("desde Arad se puede ir a:", problem.ACTIONS("Arad"))
print("Arad -> Sibiu cuesta:", problem.ACTION_COST("Arad", "Sibiu", "Sibiu"), "km")
print("¿Bucharest es el objetivo?", problem.IS_GOAL("Bucharest"))

desde Arad se puede ir a: ['Sibiu', 'Timisoara', 'Zerind']
Arad -> Sibiu cuesta: 140 km
¿Bucharest es el objetivo? True


### 3.3.1 — Los nodos del árbol de búsqueda

Tener en cuenta:

- un **estado** es una configuración del mundo (acá una ciudad);
- un **nodo** es una forma de haber llegado a un estado.

Dos nodos distintos pueden tener el mismo estado: son dos caminos distintos hasta
la misma ciudad. Por eso el nodo guarda de la historia.

```
node.STATE      el estado al que corresponde el nodo
node.PARENT     el nodo que lo genero
node.ACTION     la accion que se aplico al padre para generarlo
node.PATH_COST  el costo total del camino desde el estado inicial (g(n))
```

`failure` y `cutoff` son dos nodos particulares que los algoritmos devuelven cuando
no hay solución. Los dos tienen costo infinito.

In [6]:
import math
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class Node:
    STATE: Any
    PARENT: "Node | None" = None
    ACTION: Any = None
    PATH_COST: float = 0

    def __repr__(self):
        return f"<{self.STATE} g={self.PATH_COST:g}>"


# Nodos especiales que devuelven los algoritmos cuando no hay solucion.
failure = Node(STATE="failure", PATH_COST=math.inf)
cutoff = Node(STATE="cutoff", PATH_COST=math.inf)

Función **EXPAND**: se le da un nodo y devuelve los hijos. Es un generador (yield), así que los
hijos se crean de a uno, a medida que se piden.

In [7]:
def expand(problem, node):
    """EXPAND(problem, node) yields nodes.

    Genera los hijos de `node` aplicando cada accion posible.
    """
    s = node.STATE
    for action in problem.ACTIONS(s):
        s_prime = problem.RESULT(s, action)
        cost = node.PATH_COST + problem.ACTION_COST(s, action, s_prime)
        yield Node(STATE=s_prime, PARENT=node, ACTION=action, PATH_COST=cost)

In [8]:
raiz = Node(STATE=problem.INITIAL)
print("raiz:  ", raiz)
print("hijos: ", list(expand(problem, raiz)))

raiz:   <Arad g=0>
hijos:  [<Sibiu g=140>, <Timisoara g=118>, <Zerind g=75>]


Definimos tres funciones más para los nodos: el camino de estados, el camino de
acciones y la profundidad.

In [9]:
def path_states(node) -> list:
    """La secuencia de estados desde el inicial hasta `node`."""
    if node is None:
        return []
    return path_states(node.PARENT) + [node.STATE]


def path_actions(node) -> list:
    """La secuencia de acciones desde el inicial hasta `node`."""
    if node is None or node.PARENT is None:
        return []
    return path_actions(node.PARENT) + [node.ACTION]


def depth(node) -> int:
    """DEPTH(node): cuantas acciones hay desde el estado inicial."""
    if node is None or node.PARENT is None:
        return 0
    return 1 + depth(node.PARENT)

### 3.3.2 — La frontera

La frontera son los nodos generados pero todavía no expandidos: el borde entre lo
explorado y lo que falta. Es una cola. Vamos a definir las siguientes operaciones que serán de utilidad:

```
IS_EMPTY(frontier)   true solo si no quedan nodos
POP(frontier)        saca y devuelve un nodo
TOP(frontier)        devuelve el nodo sin sacarlo
ADD(node, frontier)  inserta el nodo en la cola
```

Vamos a ver que en los diferentes algoritmos cambia el nodo que se saca, en algunos se saca el último agregado (**LIFO**) y en otros el más antiguo (**FIFO**).

In [10]:
import heapq
from collections import deque
from itertools import count


class PriorityQueue:
    """Arriba esta el nodo de menor key(node).

    Es la frontera de best-first search: `key` es la funcion f.
    """

    def __init__(self, items=(), key=lambda node: node):
        self.key = key
        self.tie = count()      # desempata por orden de llegada
        self.heap = []
        for item in items:
            self.ADD(item)

    def ADD(self, node):
        heapq.heappush(self.heap, (self.key(node), next(self.tie), node))

    def POP(self):
        return heapq.heappop(self.heap)[2]

    def TOP(self):
        return self.heap[0][2]

    def IS_EMPTY(self):
        return not self.heap

    def nodes(self):
        """Los nodos en el orden en que van a salir. Solo para mostrar."""
        return [node for _, _, node in sorted(self.heap)]

    def __len__(self):
        return len(self.heap)


class FIFOQueue:
    """Arriba esta el nodo que entro primero.

    Es la frontera de breadth-first search: los nodos salen en el mismo
    orden en que se generaron, asi que se recorre el arbol por niveles.
    """

    def __init__(self, items=()):
        self.items = deque(items)

    def ADD(self, node):
        self.items.append(node)

    def POP(self):
        return self.items.popleft()

    def TOP(self):
        return self.items[0]

    def IS_EMPTY(self):
        return not self.items

    def nodes(self):
        return list(self.items)

    def __len__(self):
        return len(self.items)


class LIFOQueue:
    """Arriba esta el ultimo nodo que entro: una pila.

    Es la frontera de depth-first search: siempre sale el hijo recien
    generado, asi que se baja por una rama hasta el fondo.
    """

    def __init__(self, items=()):
        self.items = list(items)

    def ADD(self, node):
        self.items.append(node)

    def POP(self):
        return self.items.pop()

    def TOP(self):
        return self.items[-1]

    def IS_EMPTY(self):
        return not self.items

    def nodes(self):
        return list(reversed(self.items))

    def __len__(self):
        return len(self.items)

### Código auxiliar
Código para la visualización del problema


In [11]:
class Silent:
    """View nulo: el algoritmo no narra nada."""

    # ROUND marca que empieza una vuelta nueva: una iteracion de
    # profundidad iterativa, o un lado de la busqueda bidireccional.
    def ROUND(self, label, others=None): pass
    def POP(self, node, frontier, reached): pass
    def CHILD(self, child, reached): pass
    def NOTE(self, texto): pass
    def STEP(self, frontier, reached): pass
    def DONE(self, node, frontier, reached): pass


silent = Silent()

In [12]:
from html import escape

from IPython.display import HTML

from web import Web


def ver(view, alto=800):
    """Mete la pagina que grabo el view adentro de esta celda."""
    return HTML(f'<div><iframe srcdoc="{escape(view.html(), quote=True)}" '
                f'style="width:100%; height:{alto}px; border:1px solid #ccd2ce; '
                f'border-radius:10px" title="la busqueda paso a paso"></iframe></div>')

***


## Parte 2 — Algoritmos de búsqueda no informada
### 2.1 BEST-FIRST-SEARCH

Se elige una función **f(n)** y el algoritmo expande siempre el nodo de la frontera con **f más chica**.

`reached` es un diccionario `estado → nodo`: guarda, para cada ciudad que ya
vimos, el mejor nodo con el que llegamos. Un hijo entra a la frontera solo si es
la primera vez que vemos esa ciudad, **o** si llegamos por un camino más corto
que el que estaba registrado. 

| f(n) | algoritmo | cuándo |
|---|---|---|
| `g(n)` | costo uniforme (Dijkstra) | hoy |
| `h(n)` | greedy best-first | próxima clase |
| `g(n) + h(n)` | **A\*** | próxima clase |

El objetivo se prueba al tomar un nodo de la frontera, no al
generarlo. Eso importa: es lo que hace que el resultado sea óptimo cuando `f = g`,
y es justo lo contrario de lo que hace la búsqueda en amplitud.

In [13]:
def best_first_search(problem, f, view=silent):
    """Busca un nodo objetivo expandiendo siempre el de f minimo."""
    node = Node(STATE=problem.INITIAL)
    frontier = PriorityQueue([node], key=f)
    reached = {problem.INITIAL: node}
    while not frontier.IS_EMPTY():
        node = frontier.POP()
        view.POP(node, frontier, reached)
        if problem.IS_GOAL(node.STATE):
            view.DONE(node, frontier, reached)
            return node
        for child in expand(problem, node):
            s = child.STATE
            view.CHILD(child, reached)
            if s not in reached or child.PATH_COST < reached[s].PATH_COST:
                reached[s] = child
                frontier.ADD(child)
        view.STEP(frontier, reached)
    view.DONE(failure, frontier, reached)
    return failure

### 2.2 — Búsqueda en amplitud

La frontera es una cola FIFO. Se toman primero los nodos más antiguos, así que el
   árbol se recorre por niveles: primero todos los de profundidad 1, después los
   de profundidad 2, y así.

Se evalua si se llegó al objetivo al generar los hijos. Se corta la búsquda una vez que se alcanza el objetivo. Es de útilidad en problemas donde todas las acciónes tienen el mismo costo: cuando se encuentra el objetivo, como ya se recorrieron todos los niveles anteriores, el primer camino que aparece tiene que ser el óptimo. 

Por esto mismo, acá definimos `reached` es un **conjunto de estados**, no un diccionario de nodos.

In [14]:
def breadth_first_search(problem, view=silent):
    """Expande primero los nodos menos profundos."""
    node = Node(STATE=problem.INITIAL)
    if problem.IS_GOAL(node.STATE):
        return node
    frontier = FIFOQueue([node])
    reached = {problem.INITIAL}          # un conjunto de estados, no de nodos
    while not frontier.IS_EMPTY():
        node = frontier.POP()
        view.POP(node, frontier, reached)
        for child in expand(problem, node):
            s = child.STATE
            view.CHILD(child, reached)
            if problem.IS_GOAL(s):
                view.DONE(child, frontier, reached)
                return child
            if s not in reached:
                reached.add(s)
                frontier.ADD(child)
        view.STEP(frontier, reached)
    view.DONE(failure, frontier, reached)
    return failure

In [15]:
amplitud = Web(problem, "BREADTH-FIRST-SEARCH", "cola FIFO",
               key=depth, key_name="prof")

solucion_amplitud = breadth_first_search(problem, view=amplitud)

print(" -> ".join(path_states(solucion_amplitud)))
print("costo:", solucion_amplitud.PATH_COST)

Arad -> Sibiu -> Fagaras -> Bucharest
costo: 450


El algorítmo terminó en 5 pasos pero la solución que encontró tiene una distancia de **450 km**: encontró el camino de menos ciudades, no
el más corto. ¿Por qué?

In [16]:
#| column: screen-inset
ver(amplitud)

### 2.3 — Costo uniforme (Dijkstra)

Se usa como base el algoritmo de `best_first_search` pero con **f(n) = g(n)**: se expande el nodo con el camino de menor costo encontrado hasta ahora.

In [17]:
def g(node):
    """El costo del camino hasta el nodo, g(n)."""
    return node.PATH_COST


uniforme = Web(problem, "BEST-FIRST-SEARCH", "f(n) = g(n)",
               key=g)

solucion_uniforme = best_first_search(problem, g, view=uniforme)

print(" -> ".join(path_states(solucion_uniforme)))
print("costo:", solucion_uniforme.PATH_COST)

Arad -> Sibiu -> Rimnicu -> Pitesti -> Bucharest
costo: 418


El camino óptimo encontrado por el algoritmos es de 418 km. Se hicieron 13 expansiones en vez de 5, y encontró un camino de 4 ciudades en vez de 3, pero de menos kilómetros.

Los nodos *obsoletos* son nodos de una ciudad donde después se encontró un camino para llegar más corto.

In [18]:
#| column: screen-inset
ver(uniforme)

### 2.4 — Búsqueda en profundidad

Cambiamos la FIFO por una **LIFO**. Se toma siempre el último hijo
generado, así que la búsqueda profundiza en una "rama", en vez de un "nivel".

No hay `reached`: El uso de memoria es lineal con respecto a la profundidad en vez de exponencial. Pero puede volver a un estado ya visto. En un árbol no
pasaría, pero como el mapa es un grafo si. Definimos el método `IS_CYCLE` para cortarlo, mirando los ancestros del nodo.

In [19]:
def is_cycle(node, k=30) -> bool:
    """IS-CYCLE(node): el estado del nodo ya aparece en sus ancestros."""
    def find(ancestor, k):
        return (ancestor is not None and k > 0
                and (ancestor.STATE == node.STATE or find(ancestor.PARENT, k - 1)))
    return find(node.PARENT, k)

In [20]:
def depth_first_search(problem, view=silent):
    """Baja por una rama hasta el fondo antes de probar la siguiente."""
    frontier = LIFOQueue([Node(STATE=problem.INITIAL)])
    while not frontier.IS_EMPTY():
        node = frontier.POP()
        view.POP(node, frontier, None)
        if problem.IS_GOAL(node.STATE):
            view.DONE(node, frontier, None)
            return node
        if not is_cycle(node):
            for child in expand(problem, node):
                view.CHILD(child, None)
                frontier.ADD(child)
        else:
            view.NOTE("IS_CYCLE: esta ciudad ya esta en el camino, no se expande")
        view.STEP(frontier, None)
    view.DONE(failure, frontier, None)
    return failure

In [21]:
profundidad = Web(problem, "DEPTH-FIRST-SEARCH", "pila LIFO, sin reached",
                  key=depth, key_name="prof")

solucion_profundidad = depth_first_search(problem, view=profundidad)

print(" -> ".join(path_states(solucion_profundidad)))
print("costo:", solucion_profundidad.PATH_COST)

Arad -> Zerind -> Oradea -> Sibiu -> Rimnicu -> Pitesti -> Bucharest
costo: 575


Solución encontrada: **575 km y 7 ciudades.** 

Además nueve ciudades se expanden más de una vez y `IS_CYCLE` se dispara diez veces. Ocupamos menos memoria pero gastamos más computo.  

In [22]:
#| column: screen-inset
ver(profundidad)

### 2.5 — Profundidad limitada

Al tomar una rama y profundizar hasta que termine puede pasar que se caiga en en un espacio infinito. Una solución es ponerle un límite de pasos `l` y no expandir nada más
profundo que eso.

Solo hay que agregar este límite al algoritmo:

```python
result = failure          # <- lo que devolvemos si no encontramos
...
if depth(node) > l:       # <- el techo
    result = cutoff       # <- Marcar como corte por límite
elif not is_cycle(node):
    ...
```

Ahora tenemos un tercer valor de retorno.

Un **nodo objetivo** puede encontrar:
- **`failure`**: no hay solución, se recorrió todo lo que se podía;
- **`cutoff`**: no encontró dentro del límite, pero podría haber solución más abajo.

In [23]:
def depth_limited_search(problem, l=math.inf, view=silent):
    """Baja por una rama hasta el limite `l` antes de probar la siguiente."""
    frontier = LIFOQueue([Node(STATE=problem.INITIAL)])
    result = failure
    while not frontier.IS_EMPTY():
        node = frontier.POP()
        view.POP(node, frontier, None)
        if problem.IS_GOAL(node.STATE):
            view.DONE(node, frontier, None)
            return node
        if depth(node) > l:
            result = cutoff
            view.NOTE(f"pasa el limite l = {l}: no se expande")
        elif not is_cycle(node):
            for child in expand(problem, node):
                view.CHILD(child, None)
                frontier.ADD(child)
        else:
            view.NOTE("IS_CYCLE: esta ciudad ya esta en el camino, no se expande")
        view.STEP(frontier, None)
    view.DONE(result, frontier, None)
    return result

Con `l = ∞` es exactamente la búsqueda en profundidad anterior.

In [24]:
print(path_states(depth_limited_search(problem, math.inf)))
print(path_states(depth_first_search(problem)))

['Arad', 'Zerind', 'Oradea', 'Sibiu', 'Rimnicu', 'Pitesti', 'Bucharest']
['Arad', 'Zerind', 'Oradea', 'Sibiu', 'Rimnicu', 'Pitesti', 'Bucharest']


Con límites chicos podemos ver el cutoff:

In [25]:
for l in range(6):
    r = depth_limited_search(problem, l)
    if r is cutoff:
        print(f"l = {l}   cutoff")
    else:
        print(f"l = {l}   costo {r.PATH_COST:>3g}   {' -> '.join(path_states(r))}")

l = 0   cutoff
l = 1   cutoff
l = 2   costo 450   Arad -> Sibiu -> Fagaras -> Bucharest
l = 3   costo 418   Arad -> Sibiu -> Rimnicu -> Pitesti -> Bucharest
l = 4   costo 607   Arad -> Zerind -> Oradea -> Sibiu -> Fagaras -> Bucharest
l = 5   costo 575   Arad -> Zerind -> Oradea -> Sibiu -> Rimnicu -> Pitesti -> Bucharest


Subir el límite puede dar una peor solución: Teniendo más libertad para bajar por una misma rama puede terminar encontrando en una de las primera un camino que cumpla el objetivo pero que no sea el óptimo.

In [26]:
#| column: screen-inset
LIMITE = 2
limitada = Web(problem, "DEPTH-LIMITED-SEARCH", "pila LIFO, l = 1",
               key=depth, key_name="prof")

depth_limited_search(problem, LIMITE, view=limitada)
ver(limitada)

### 2.6 — Profundidad iterativa

Búsqueda en profundidad limitada, pero probando `l = 0, 1, 2, ...` hasta encontrar.
Se queda con lo mejor de los dos lados: la memoria de la búsqueda en profundidad
—lineal con la profundidad— y la garantía de la búsqueda en amplitud de encontrar
primero la solución más superficial.

Parece un desperdicio rehacer el trabajo en cada vuelta, pero no lo es: el último
nivel tiene más nodos que todos los anteriores juntos, así que la última vuelta
pesa más que todas las anteriores sumadas.


In [27]:
def iterative_deepening_search(problem, view=silent):
    """Prueba con limite 0, 1, 2, ... hasta encontrar."""
    # En el libro esta variable se llama `depth`; aca `l`, para no pisar
    # la funcion DEPTH(node).
    for l in count():
        view.ROUND(f"l = {l}")
        result = depth_limited_search(problem, l, view=view)
        if result is not cutoff:
            return result

In [28]:
iterativa = Web(problem, "ITERATIVE-DEEPENING-SEARCH",
                "depth-limited con l = 0, 1, 2, ...",
                key=depth, key_name="prof")

solucion_iterativa = iterative_deepening_search(problem, view=iterativa)

print(" -> ".join(path_states(solucion_iterativa)))
print("costo:", solucion_iterativa.PATH_COST)

print()
for l in range(3):
    vuelta = Web(problem, "", "", key=depth)
    depth_limited_search(problem, l, view=vuelta)
    print(f"l = {l:<5} {vuelta.pops:>3} expansiones")
print(f"{'total':<9} {iterativa.pops:>3}")

Arad -> Sibiu -> Fagaras -> Bucharest
costo: 450

l = 0       4 expansiones
l = 1      12 expansiones
l = 2      22 expansiones
total      38


Devolvió lo mismo que la búsqueda en amplitud: 450 km y 3 ciudades. La solución de menos pasos. ¿Por qué?


In [29]:
#| column: screen-inset
ver(iterativa)

### 2.7 — Búsqueda bidireccional

Dos búsquedas a la vez: una que sale de Arad hacia adelante y otra que sale de
Bucharest hacia atrás, hasta que se cruzan en alguna ciudad del medio.

Tres funciones auxiliares antes del algoritmo:

In [30]:
def terminated(solution, frontier_f, frontier_b, f_f, f_b):
    """TERMINATED: ya no puede aparecer una solucion mejor.

    Si los dos nodos de arriba juntos ya cuestan mas que el mejor camino
    que tenemos, cualquier camino que quede por armar va a ser peor.
    Mientras no haya solucion, PATH_COST de `failure` es infinito y esto
    nunca corta.
    """
    if frontier_f.IS_EMPTY() or frontier_b.IS_EMPTY():
        return True
    return f_f(frontier_f.TOP()) + f_b(frontier_b.TOP()) > solution.PATH_COST


def join_nodes(dir, node1, node2):
    """JOIN-NODES: pega el camino de ida con el de vuelta dado vuelta.

    Los dos nodos estan en la misma ciudad. El de ida trae el camino
    desde INITIAL; el de vuelta trae el camino hasta GOAL, pero armado
    al reves, asi que hay que rehacerlo hacia adelante.
    """
    ida, vuelta = (node1, node2) if dir == "F" else (node2, node1)
    node = ida
    while vuelta.PARENT is not None:
        paso = vuelta.PATH_COST - vuelta.PARENT.PATH_COST
        # El camino de vuelta sabe por que ciudades pasa, pero no con
        # que accion se llega a cada una yendo para adelante.
        node = Node(STATE=vuelta.PARENT.STATE, PARENT=node, ACTION=None,
                    PATH_COST=node.PATH_COST + paso)
        vuelta = vuelta.PARENT
    return node


def proceed(dir, problem, frontier, reached, reached2, solution, view=silent):
    """Expande un nodo de un lado y lo compara contra el otro lado.

    `dir` es F si el que avanza es el de ida, B si es el de vuelta.
    """
    node = frontier.POP()
    view.POP(node, frontier, reached)
    for child in expand(problem, node):
        s = child.STATE
        view.CHILD(child, reached)
        if s not in reached or child.PATH_COST < reached[s].PATH_COST:
            reached[s] = child
            frontier.ADD(child)
            if s in reached2:
                # Los dos lados llegaron a la misma ciudad: hay camino
                # entero. Puede que no sea el mejor, asi que se compara.
                solution2 = join_nodes(dir, child, reached2[s])
                if solution2.PATH_COST < solution.PATH_COST:
                    solution = solution2
                    view.NOTE(f"se cruzan en {s}: "
                              f"camino entero de {solution.PATH_COST:g}")
    view.STEP(frontier, reached)
    return solution

Y el algoritmo. En cada vuelta avanza el lado cuyo nodo en la cola tiene el costo más bajo.

In [31]:
def bibf_search(problem_f, f_f, problem_b, f_b, view=silent):
    """Best-first por los dos lados a la vez."""
    node_f = Node(STATE=problem_f.INITIAL)      # nodo del estado inicial
    node_b = Node(STATE=problem_b.INITIAL)      # nodo del estado objetivo
    frontier_f = PriorityQueue([node_f], key=f_f)
    frontier_b = PriorityQueue([node_b], key=f_b)
    reached_f = {node_f.STATE: node_f}
    reached_b = {node_b.STATE: node_b}
    solution = failure
    while not terminated(solution, frontier_f, frontier_b, f_f, f_b):
        if f_f(frontier_f.TOP()) < f_b(frontier_b.TOP()):
            view.ROUND("adelante", reached_b)
            solution = proceed("F", problem_f, frontier_f,
                               reached_f, reached_b, solution, view)
        else:
            view.ROUND("atras", reached_f)
            solution = proceed("B", problem_b, frontier_b,
                               reached_b, reached_f, solution, view)
    # El ultimo cuadro muestra los dos lados juntos.
    view.ROUND("los dos lados", reached_b)
    view.DONE(solution, frontier_f, reached_f)
    return solution


def bidirectional_search(problem_f, problem_b, view=silent):
    """Bidireccional con f(n) = g(n) de los dos lados: uniform-cost."""
    return bibf_search(problem_f, g, problem_b, g, view=view)

In [32]:
problem_b = RouteProblem("Bucharest", "Arad", map=romania)

bidireccional = Web(problem, "BIBF-SEARCH", "dos frentes, f(n) = g(n)",
                    key=g, key_name="g")

solucion_bidireccional = bidirectional_search(problem, problem_b, view=bidireccional)

print(" -> ".join(path_states(solucion_bidireccional)))
print("costo:", solucion_bidireccional.PATH_COST)

Arad -> Sibiu -> Rimnicu -> Pitesti -> Bucharest
costo: 418


Solución encontrada: 418 km, el óptimo, en 11 expansiones.

In [33]:
#| column: screen-inset
ver(bidireccional)

***
## 2.8 — Comparación 

`b` es el factor de ramificación, `d` la profundidad de la solución más
superficial, `m` la profundidad máxima del árbol, `ℓ` el límite de profundidad.

| criterio | búsqueda en amplitud | costo uniforme | búsqueda en profundidad | profundidad limitada | profundidad iterativa | búsqueda bidireccional |
|---|:---:|:---:|:---:|:---:|:---:|:---:|
| ¿completo? | sí ¹ | sí ^(1,2) | no | no | sí ¹ | sí ^(1,4) |
| ¿costo óptimo? | sí ³ | **sí** | no | no | sí ³ | sí ^(3,4) |
| tiempo | O(b^d) | O(b^(1+⌊C\*/ε⌋)) | O(b^m) | O(b^ℓ) | O(b^d) | O(b^(d/2)) |
| espacio | O(b^d) | O(b^(1+⌊C\*/ε⌋)) | **O(bm)** | **O(bℓ)** | **O(bd)** | O(b^(d/2)) |

¹ completo si `b` es finito y el espacio de estados o tiene solución o es finito.
² completo si todos los costos de acción son ≥ ε > 0.
³ óptimo en costo si todos los costos de acción son iguales.
⁴ si los dos lados usan búsqueda en amplitud o costo uniforme.


In [34]:
resultados = [
    ("búsqueda en amplitud",    solucion_amplitud,      amplitud),
    ("costo uniforme",          solucion_uniforme,      uniforme),
    ("búsqueda en profundidad", solucion_profundidad,   profundidad),
    ("profundidad iterativa",   solucion_iterativa,     iterativa),
    ("búsqueda bidireccional",  solucion_bidireccional, bidireccional),
]

print(f"{'algoritmo':<25}{'costo':>7}{'ciudades':>10}{'expande':>9}   camino")
print("-" * 99)
for nombre, nodo, view in resultados:
    estados = path_states(nodo)
    print(f"{nombre:<25}{nodo.PATH_COST:>7g}{len(estados):>10}{view.pops:>9}   "
          f"{' '.join(estados[1:-1])}")

algoritmo                  costo  ciudades  expande   camino
---------------------------------------------------------------------------------------------------
búsqueda en amplitud         450         4        5   Sibiu Fagaras
costo uniforme               418         5       13   Sibiu Rimnicu Pitesti
búsqueda en profundidad      575         7       22   Zerind Oradea Sibiu Rimnicu Pitesti
profundidad iterativa        450         4       38   Sibiu Fagaras
búsqueda bidireccional       418         5       11   Sibiu Rimnicu Pitesti


Observaciones:

- **búsqueda en amplitud** y **profundidad iterativa** minimizan *ciudades* → 450 km
- **costo uniforme** y **búsqueda bidireccional** minimizan *kilómetros* → 418 km
- **búsqueda en profundidad** no minimiza nada → 575 km, pero optimiza uso de memoria.